<a href="https://colab.research.google.com/github/simoncurchod-lang/2026_Intro_Python/blob/main/live/part-I/1.2-data-structures-and-control-flow-environmental-exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.2-data-structures-and-control-flow-environmental-exercises.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [1]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Environmental Exercises

**ℹ️ How to use these exercises**

Attempt each from scratch in the empty code cell. Write plain functions with clear names, and avoid a mutable default argument.

## Exercise 1: A year of station data, continued

Exercises 1 and 2 are the main exercises of this subchapter.

In subchapter 1.1 you fetched this file, worked out its structure, and converted a single
record to °C. You did not yet have the tools to handle all of it at once — now you do.

The goal: turn a year of daily maximum temperatures into a monthly summary, and write that
summary out as a new file. The input file has the same three columns as in Exercise 1 of the
previous subchapter's environmental exercises: the station code, the day as `dd.mm.yy`, and the daily maximum temperature
in degrees Fahrenheit. Your monthly summary will have its own columns: `month`, `n_obs`
(the number of observations), and `mean_temp_celsius`.

In [2]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

data_file = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/station_iib_daily_max_temp_2022.csv",
    known_hash="sha256:034755fb289b4e157e5d029995481786a359eb25a80df62c09ee83d063013144",
    fname="station_iib_daily_max_temp_2022.csv",
    path=pooch.os_cache("mlees"),
)
print("data cached at:", data_file)

data cached at: /root/.cache/mlees/station_iib_daily_max_temp_2022.csv


**Step 1.** Write a function `fahrenheit_to_celsius(temp_fahrenheit)` that returns the
temperature in °C. Test it on the freezing point of water (32 °F should give 0.0 °C) and on
212 °F. A one-line docstring is optional here (see Exercise 11 of the practice exercises).

You wrote this conversion inline in 1.1. Steps 2 and 4 both need it again; with a function,
they call the same tested code instead of repeating the arithmetic.

In [25]:
# Your solution here
# Hint: °C = (°F - 32) * 5 / 9
# Hint (optional, see Exercise 11 of the practice exercises): a docstring goes on the first line of the function body,
# in triple quotes
def fahrenheit_to_celsius(temp_fahrenheit):
    temp_cel = (temp_fahrenheit-32)*5/9
    return temp_cel

0.0

**Step 2.** Open the file, read all lines, and loop over the records (skipping the header).
For each one, extract the month and the temperature in °C. Print the first three as a check.

Sixteen days in this file have an empty temperature field — a real gap in the station's record,
late October to mid-November. `float("")` raises `ValueError`, so your loop has to decide what to
do with those days before it can get to the end of the file. Skip them, and count how many you
skipped. After the loop, print how many records you converted and how many you skipped; the two
should add up to 365.

In [54]:
# Your solution here
# Hint: open it as in 1.1, with Path(data_file).open("r", encoding="utf-8") in a with block
# Hint: lines[1:] skips the header row
# Hint: .strip().split(",") gives you the three fields
# Hint: the month is the middle piece of the day field — split it on "."
# Hint: use a counter that you update with += inside the loop
# Hint: a day with no reading has fields[2] == "" — `continue` moves on to the next line
from pathlib import Path

with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines = fhandler.readlines()

temp= []
month = []
skipped = 0
for line in lines[1:]:
    suppr_row = line.strip().split(",")
    if suppr_row[2] == "":
      skipped += 1
      continue
    inter = suppr_row[1].split(".")
    month.append(int(inter[1]))
    temp.append(fahrenheit_to_celsius(float(suppr_row[2])))

month_temp = zip(month, temp)
month_temp = list(month_temp)
print(len(temp), skipped, len(temp) + skipped)

#month_temp.insert(0,("month", "temp [C°]"))
print(month_temp)

349 16 365
[(1, -9.61111111111111), (1, -13.0), (1, -4.222222222222223), (1, 0.9999999999999984), (1, -5.777777777777777), (1, -17.0), (1, -13.0), (1, -0.6111111111111119), (1, -4.111111111111111), (1, -10.0), (1, 4.111111111111111), (1, 5.499999999999999), (1, 6.111111111111111), (1, 1.6111111111111103), (1, -7.5), (1, -1.722222222222223), (1, -1.6111111111111103), (1, 3.4999999999999982), (1, -6.111111111111111), (1, -15.38888888888889), (1, -9.11111111111111), (1, -4.222222222222223), (1, -7.000000000000001), (1, -3.277777777777777), (1, -15.5), (1, -8.0), (1, 1.7777777777777795), (1, -9.5), (1, -1.2222222222222219), (1, -4.0), (1, 12.777777777777779), (2, 3.277777777777777), (2, -10.61111111111111), (2, -11.38888888888889), (2, -8.88888888888889), (2, -2.000000000000001), (2, 2.6111111111111125), (2, -5.222222222222221), (2, 5.277777777777778), (2, 4.111111111111111), (2, 3.7222222222222237), (2, 4.388888888888888), (2, -10.11111111111111), (2, -11.0), (2, -2.1111111111111116), (2,

**Step 3.** Build a dictionary mapping each month to the number of observations in it.

You do not know in advance which months are present, so the dictionary has to grow as you go.
Keep skipping the days with no reading: a day the sensor did not record is not an observation, so
October and November should come out short of their full length.

In [59]:
# Your solution here
# Hint: start with an empty dict, counts = {}
# Hint: counts.get(month, 0) returns 0 for a month you have not seen yet
# Hint: counts[month] = counts.get(month, 0) + 1

counts = {}
for m in month:
    counts[m] = counts.get(m, 0) + 1
counts.items()

dict_items([(1, 31), (2, 28), (3, 31), (4, 30), (5, 31), (6, 30), (7, 31), (8, 31), (9, 30), (10, 24), (11, 21), (12, 31)])

**Step 4.** Build a second dictionary mapping each month to its **mean** daily maximum
temperature in °C. Print it with each mean rounded to two decimals.

In [63]:
# Your solution here
# Hint: accumulate a running total per month in the same loop as the counts
# Hint: a second loop over the totals turns each total into a mean
# Hint: reuse your function from step 1 — do not repeat the conversion arithmetic
summarise = {}
for m, t in zip(month, temp):
    summarise[m] = counts.get(m, 0) + t

mean = {m: summarise[m]/counts[m] for m in summarise}
mean.items()

dict_items([(1, 1.4121863799283154), (2, 1.4563492063492063), (3, 1.593189964157706), (4, 1.6333333333333333), (5, 1.8548387096774193), (6, 2.009259259259259), (7, 1.903225806451613), (8, 1.9193548387096775), (9, 1.7166666666666666), (10, 1.775462962962963), (11, 0.7724867724867724), (12, 1.0232974910394266)])

**Step 5.** Find the warmest month and print it with its mean temperature.

Do this with a loop and a comparison, keeping track of the best value seen so far, as you did for
the warmest day in Exercise 6 of the practice exercises. A dictionary is looked up by key, not by position, so there is no
`[0]` to start from: start from `None` instead.

In [7]:
# Your solution here
# Hint: initialise warmest_month = None and warmest_mean = None before the loop
# Hint: if warmest_mean is None, take this month; elif its mean is larger, take it instead
warmest_month = None
warmest_mean = None

for w in mean
    if warmest_mean

**Step 6.** Write your monthly summary to `_files/monthly_means.csv`, with the header
`month,n_obs,mean_temp_celsius` and one line per month, creating the `_files` folder first as you
did in 1.1. Then read the file back and print it, to confirm it says what you meant it to say.

In [8]:
# Your solution here
# Hint: out_path = Path("_files") / "monthly_means.csv" builds the path, and
# out_path.parent.mkdir(exist_ok=True) creates the _files folder if it is missing
# Hint: open with mode "w" inside a with block, and write the header first
# Hint: build each line with an f-string, and remember the "\n"
# Hint: f"{value:.2f}" keeps the file tidy

**Step 7 (optional).** This step uses `try`/`except` from the going-deeper box in the lecture. Real files contain junk, and you have already met one kind of it — the empty
temperature field. Suppose some lines are truncated as well, or hold a temperature that is not a
number at all. Modify your reading loop so that it skips any line that does not have three fields
or whose temperature will not convert to a number, counting how many lines it skipped.

Test it on this list before applying it to the real file:

```python
test_lines = [
    "IIB,01.01.22,30.2",
    "IIB,02.01.22",
    "IIB,03.01.22,notanumber",
    "IIB,04.01.22,33.4",
]
```

You should keep 2 and skip 2.

In [9]:
# Your solution here
# Hint: len(fields) != 3 catches the truncated line — use continue
# Hint: wrap the float() call in try/except ValueError to catch the bad number
# Hint: continue inside a for loop moves straight to the next line

**ℹ️ What you just did**

You wrote a reusable function, looped over a whole year of records, built dictionaries whose
keys you did not know in advance, aggregated by group, and wrote a derived dataset back to disk
— with a loop that survives malformed input. This is the shape of nearly every data-processing
script you will write. In subchapter 1.5, `pandas` will do the grouping in one line; the
reasoning will be exactly the same.

## Exercise 2: The solar system

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/nasa-solar-system-montage.jpg" alt="The eight planets and the Moon photographed by spacecraft, arranged diagonally against black" width="100%">

<em>The eight planets and the Moon, as photographed by Mariner 10, Magellan, Galileo, Mars Global
Surveyor, Cassini and the Voyager spacecraft. The inner bodies are roughly to scale with each other, as are the outer ones, but the
two groups are not to scale with one another. Image
<a href="https://www.jpl.nasa.gov/images/pia03153-solar-system-montage-high-resolution-2001-version/">NASA/JPL (PIA03153)</a>, public domain.</em>


This is the long exercise for this subchapter, and the only one in it whose data is not
environmental. Eight planets and their masses are few enough to type by hand, and familiar enough
that a wrong answer is easy to spot, which makes them a good first test of lists, dictionaries and
functions together.

Work through sections A to C in order: section C writes functions against the dictionary you
build in section B. You will pick the exercise up again in 1.7, once classes are available, and
turn the dictionary into a `Planet` type.

The masses come from NASA's
[planetary fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/). Use units of
10^24 kg throughout, so that Earth is 5.97 and Jupiter is 1898 — the variable names below carry
that unit, which is what makes a bare number like `1898` readable three cells later.

### A: lists and loops

**Q1) Create a list with the names of every planet in the solar system, in order.**

Name it `planets` and use lowercase names, so it starts `["mercury", "venus", ...]`.

In [10]:
# Q1: create your list here

**Q2) Have Python tell you how many planets there are by examining your list.**

You should get 8. Do not type the number — read it off the list.

In [11]:
# Q2: count the planets
# Hint: len() reads the length off the list itself

**Q3) Use slicing to display the first four planets — the rocky ones.**

In [12]:
# Q3: the rocky planets

**Q4) Iterate through your planets and print the planet name only if it ends in "s".**

You should see `venus`, `mars` and `uranus`, and nothing else.

Hint: the last letter of a name is `planet[-1]`. Use `continue` for the names that do not match,
so the loop body says explicitly what it skips.

In [13]:
# Q4: print the planets whose name ends in s

### B: dictionary

**Q5) Create a dictionary that maps each planet name to its mass.**

Call it `planet_masses_1e24kg`, and read the values off the
[NASA fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/) in units of 10^24 kg. Every
value has to be in that one unit: entered as 1.898 (in 10^27 kg), Jupiter would come out 1000
times too light in every comparison and ratio below.

In [14]:
# Q5: build the dictionary here

**Q6) Use your dictionary to look up Earth's mass.**

In [15]:
# Q6: look up Earth

**Q7) Loop through the dictionary and build a list of every planet heavier than
100 x 10^24 kg.**

You should end up with `["jupiter", "saturn", "neptune"]` — uranus, at 86.8, does not make it.

Hint: `.items()` gives you the name and the mass together on each pass, as in 1.2's lecture.

In [16]:
# Q7: the planets heavier than 100e24 kg

**Q8) Add Pluto to your dictionary.**

Its mass is 0.0130 x 10^24 kg. Print the names afterwards, looping with `.items()`, to confirm it
landed.

Pluto has not been a planet since 2006. Adding it still takes one assignment, and none of the code
that reads the other eight entries has to change.

In [17]:
# Q8: add Pluto, then print the names

### C: functions

**Q9) Write a function that converts a planetary mass to Earth masses.**

Jupiter is about 1898 x 10^24 kg, which is about 318 Earth masses:

$$M_{\text{Jupiter}} \approx 1898 \times 10^{24}\,\text{kg} \approx 318\,M_{\text{Earth}}$$

Call it `to_earth_masses(mass_1e24kg)` and check it against Jupiter — you should get
317.92294807370183.

In [18]:
# Q9: write to_earth_masses here, then test it on Jupiter

**Q10) Now write a single function that converts to either Earth masses or Jupiter
masses, depending on a keyword argument.**

Call it `to_planet_masses(mass_1e24kg, reference="jupiter")`. Converting Jupiter's own mass should
give 317.923 with `reference="earth"` and exactly 1.0 with `reference="jupiter"`.

With the default argument, one function covers both conversions. Jupiter in Jupiter masses has
to come out as exactly 1.0; any other value means the function divides by the wrong mass.

In [19]:
# Q10: write to_planet_masses here, then test both references on Jupiter

**Q11) Write a function that returns two values.**

`mass_in_two_units(mass_1e24kg, reference)` should return the mass in Earth masses *and* in masses
of whatever reference planet you pass. Test it on Jupiter with `reference="mars"`.

Hint: `return a, b` returns a tuple, and `first, second = f(...)` unpacks it — both from 1.2's
lecture.

In [20]:
# Q11: write mass_in_two_units here, then test it on Jupiter against mars

**Bonus.** Convert Neptune's mass to Jupiter masses with your Q10 function. Then pass that result
to your Q11 function with `reference="jupiter"`, and compare the second value it returns with the
Q10 result. Are they the same? If not, work out which function is being given a number in the
wrong unit, and what it should have been given instead — this is the mistake the `_1e24kg` suffix
exists to prevent.

In [21]:
# Bonus: Neptune, out and back

**ℹ️ What you just did**

You built a list and read its length rather than counting it, sliced a subset out of it, and
looped over it with a condition. You built a dictionary keyed by name, looked a value up, filtered
it into a new list, and extended it. Then you wrote three functions over that dictionary: one
fixed, one with a default argument that collapses two functions into one, and one returning a pair.

That is the whole of this subchapter applied to one dataset. In 1.7 you will come back and replace
the dictionary with a class, so that a planet's name and mass travel together as one object
instead of as a key and a value that happen to line up.